# Statistical Data Assumptions

Formal tests of the statistical assumptions that underlie downstream modelling choices.
Results here justify (or challenge) decisions documented in `AT_comments.md`.

Covers:
1. Normality (Shapiro-Wilk, Q-Q plots)
2. Homoscedasticity (Levene's test)
3. Multicollinearity (Variance Inflation Factor)
4. Independence of targets

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

sys.path.insert(0, str(Path().resolve()))
from utils.preprocessing import FEATURE_NAMES, TARGETS, build_features, load_data

df = load_data()
X = build_features(df)
print('Dataset shape:', df.shape)
print('Engineered features:', FEATURE_NAMES)

## 1. Normality — Shapiro-Wilk test

Note: with n=5000, Shapiro-Wilk has very high power and will reject normality
for even small deviations. Interpret alongside effect size (skewness, kurtosis).

In [ ]:
# Shapiro-Wilk on a subsample (full n=5000 is above the recommended limit)
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=500, replace=False)
X_sample = X.iloc[sample_idx]

rows = []
for col in X.columns:
    stat, p = stats.shapiro(X_sample[col])
    sk = float(stats.skew(X[col]))
    kurt = float(stats.kurtosis(X[col]))
    rows.append({'feature': col, 'W': round(stat, 4), 'p_value': round(p, 4),
                 'normal': p > 0.05, 'skewness': round(sk, 3), 'kurtosis': round(kurt, 3)})

normality_df = pd.DataFrame(rows).set_index('feature')
normality_df

## 2. Q-Q plots

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(X.columns):
    sm.qqplot(X[col].values, line='s', ax=axes[i], alpha=0.3, markersize=2)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Q-Q Plots — Engineered Features', y=1.01)
plt.tight_layout()
plt.show()

## 3. Homoscedasticity — Levene's test

Tests whether feature variance is equal across the two classes of each target.
Violation is expected for Income-driven features (wealthier clients have more spread).

In [ ]:
for target in TARGETS:
    print(f'\nLevene test — groups: {target}=0 vs {target}=1')
    rows = []
    group0 = X[df[target] == 0]
    group1 = X[df[target] == 1]
    for col in X.columns:
        stat, p = stats.levene(group0[col], group1[col])
        rows.append({'feature': col, 'stat': round(stat, 3), 'p_value': round(p, 4),
                     'homoscedastic': p > 0.05})
    print(pd.DataFrame(rows).set_index('feature').to_string())

## 4. Multicollinearity — Variance Inflation Factor (VIF)

VIF > 10 indicates severe multicollinearity. Age, Age_sq, Age_x_Wealth are
**intentionally** collinear (documented in AT_comments.md) — tree models are invariant to this.

In [ ]:
X_const = sm.add_constant(X)
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [
    variance_inflation_factor(X_const.values, i + 1)
    for i in range(X.shape[1])
]
vif_data['high_vif'] = vif_data['VIF'] > 10
vif_data = vif_data.sort_values('VIF', ascending=False)
print(vif_data.to_string(index=False))

## 5. Target independence

Pearson correlation and chi-squared test between the two binary targets.

In [ ]:
r, p_pearson = stats.pearsonr(df['IncomeInvestment'], df['AccumulationInvestment'])
print(f'Pearson r = {r:.4f}  (p={p_pearson:.4f})')

contingency = pd.crosstab(df['IncomeInvestment'], df['AccumulationInvestment'])
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi-squared = {chi2:.3f}  df={dof}  p={p_chi2:.4f}')
print()
print('Contingency table:')
print(contingency)

In [ ]:
# --- Additional assumption tests go here ---